# 교안 01-3: 브라우저를 조작하는 MCP 서버 붙이기 (Playwright)

## 핵심 목표

브라우저 서버로 무한 스크롤 페이지를 실제로 움직여, **주소만으로는 닿지 않는** 이미지를 모은다.

## 학습 순서

1. Playwright 서버 연결과 도구 20여 개
2. 무한 스크롤 페이지 열기(`browser_navigate`)와 화면 구조 스냅샷(`browser_snapshot`)
3. 스크롤하며 이미지 모으기(`browser_press_key`·`browser_wait_for`·`browser_evaluate`)
4. 필요한 도구만 골라 붙인 에이전트가 스스로 스크롤해 이미지 모으기

## 이 서버가 앞의 넷과 다른 점: 상태를 가진다

파일시스템·검색·DB·코드 실행 서버는 **상태가 없었습니다**. 호출할 때마다 새로 붙어도 결과가 같았습니다.
브라우저 서버는 다릅니다. **열어 둔 페이지가 다음 호출까지 남아 있어야** `navigate → snapshot → 스크롤` 이 이어집니다.
그래서 이 노트북만 세션을 **직접 열고 닫습니다**(자세한 이유는 아래에서).

## 쓰는 MCP 서버와 공식 문서

| 서버 | 실행 | 전송 | 공식 문서 |
|---|---|---|---|
| 브라우저 조작 `@playwright/mcp` | `npx` | stdio | https://github.com/microsoft/playwright-mcp |

## 준비물

- **Node.js**(`npx -v`)
- 브라우저 실행 파일이 한 번은 설치돼 있어야 합니다(실습자료 폴더에서, 약 150MB).

```bash
npx playwright install chromium
```

- **에이전트를 만드는 절에서 `OPENAI_API_KEY`** 가 필요합니다(일차 폴더의 `.env`).

---
## 준비

In [ ]:
import sys
from pathlib import Path

# 노트북에는 __file__ 이 없다. 주피터는 노트북이 있는 폴더를 작업 폴더로 잡아 주므로 그 위가 일차 폴더다.
DAY_DIR = Path.cwd().parent        # 일차 폴더(day21). 아래 경로들의 기준점
sys.path.append(str(DAY_DIR))   # 일차 폴더의 utils.py 를 쓴다

from langchain.agents import create_agent
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain_mcp_adapters.tools import load_mcp_tools
from langchain_openai import ChatOpenAI

from utils import block_text, load_api_key, print_trajectory, result_value

OUTPUT_DIR = DAY_DIR / "output"   # 브라우저 서버가 남기는 파일을 모아 둘 곳
load_api_key(DAY_DIR)             # 모델을 부르는 절이 있으므로 키를 맨 앞에서 확인한다

# 실습에 쓸 페이지. 스크래핑 연습용으로 공개된 무한 스크롤 페이지다.
# 화면을 내릴 때마다 상품 이미지가 12장씩 더 붙는다.
SITE = "https://www.scrapingcourse.com/infinite-scrolling"

print("준비 완료. 결과 폴더:", OUTPUT_DIR)

---
## 서버 설정: 인자가 네 개나 붙는 이유

이 서버는 브라우저를 다루는 만큼 **어떻게 띄울지 선택지**가 많습니다. 그 선택을 인자로 받습니다.

| 키 | 값 | 뜻 |
|---|---|---|
| `command` | `"npx"` | 서버를 띄울 실행기(Node 패키지) |
| `args[0]` | `"-y"` | 설치 여부를 묻지 않고 진행 |
| `args[1]` | `"@playwright/mcp@latest"` | 띄울 서버 패키지 이름. `@latest` 는 **항상 최신판**을 받으라는 뜻 |
| `args[2]` | `"--isolated"` | 쿠키·로그인을 남기지 않는 **임시 프로필**로 띄운다. 내 크롬 프로필을 건드리지 않는다 |
| `args[3:5]` | `"--output-dir", str(OUTPUT_DIR)` | 스냅샷·스크린샷 **파일을 남길 폴더** |
| `transport` | `"stdio"` | 자식 프로세스로 띄우고 표준입출력으로 대화 |

여기에 **`"--headless"`** 를 더 넣으면 **창 없이** 백그라운드로 돕니다. 서버·자동화 환경에서는 그렇게 씁니다.
이 실습에서는 넣지 않습니다. **브라우저가 실제로 움직이는 것을 눈으로 보는 것**이 이 단원의 핵심이기 때문입니다.
페이지가 열리고 화면이 내려가고 이미지가 붙는 과정을 보고 나면, 뒤에 나오는 도구 이름들이 무엇을 하는지 훨씬 분명해집니다.

`--isolated` 가 특히 중요합니다. 자동화가 내 로그인 세션을 그대로 쓰면 **의도치 않은 계정 조작**이 가능해집니다.
실습에서는 항상 격리된 임시 프로필로 띄웁니다.

In [ ]:
# 브라우저 조작 서버: 페이지를 열고 화면 구조를 읽고 클릭·입력하는 도구를 내준다.
PLAYWRIGHT = {
    "command": "npx",                              # Node 패키지 실행기
    "args": ["-y", "@playwright/mcp@latest",       # 띄울 서버 패키지 이름
             # "--headless",                       # 이 줄을 살리면 창 없이 돈다(서버·자동화 환경용)
             "--isolated",                         # 쿠키·로그인을 남기지 않는 임시 프로필로 띄운다
             "--output-dir", str(OUTPUT_DIR)],     # 스냅샷·스크린샷 파일을 남길 폴더
    "transport": "stdio",                          # 내 컴퓨터에 프로세스로 띄운다
}

---
## 1. 브라우저 서버에 붙기: 이번엔 세션을 직접 열어 둔다

앞의 네 노트북은 `await client.get_tools(...)` 를 썼습니다. 그 방식은 **도구를 부를 때마다 서버에 새로 붙습니다**.
상태가 없는 서버에서는 문제가 없었지만, 브라우저 서버에서는 **열어 둔 페이지가 매번 사라져** 스크롤이 이어지지 않습니다.

`.py` 판은 `async with client.session("web") as session:` 블록 안에서 전부 처리했습니다.
노트북은 셀이 끝나면 그 블록이 닫히므로, **열고 닫는 일을 우리가 직접** 합니다.

```python
ctx = client.session("web")          # 세션을 만들 준비
session = await ctx.__aenter__()     # 연다(= async with 의 시작)
...                                  # 여러 셀에 걸쳐 쓴다
await ctx.__aexit__(None, None, None)  # 닫는다(= async with 의 끝)
```

`__aenter__`·`__aexit__` 는 `async with` 가 안에서 부르는 바로 그 함수입니다.
평소에는 `async with` 로 쓰는 편이 안전하지만, **셀 경계를 넘겨야 할 때**는 이렇게 직접 부릅니다.
마지막 절에서 반드시 닫아 브라우저 프로세스를 정리합니다.

In [ ]:
print("서버를 띄우는 중입니다(첫 실행은 오래 걸립니다)...")
client = MultiServerMCPClient({"web": PLAYWRIGHT})

ctx = client.session("web")          # 세션을 셀 경계 너머까지 살려 두려고 직접 연다
session = await ctx.__aenter__()
tools = await load_mcp_tools(session)
by_name = {tool.name: tool for tool in tools}

# 이 서버가 내주는 도구 이름을 한 줄로 훑는다. 뒤에서 여기서 고른 이름만 에이전트에 넘긴다.
print(f"도구 {len(tools)}개")
print(" ", ", ".join(sorted(by_name)))

---
## 2. 페이지 열기와 화면 읽기

`browser_navigate` 는 주소 하나를 받아 브라우저로 그 페이지를 엽니다. 열어 둔 페이지는 **다음 호출까지 그대로** 남습니다.

이 셀을 실행하면 **크롬 창이 실제로 뜹니다**(`--headless` 를 넣지 않았으므로).
창을 닫지 마세요. 다음 셀들이 그 창을 계속 씁니다. 정리는 마지막 절의 닫기 셀이 합니다.

In [ ]:
# navigate 결과에는 "### Result" 절이 없다. result_value 는 그 절을 꺼내는 함수라 여기엔 맞지 않는다.
print(block_text(await by_name["browser_navigate"].ainvoke({"url": SITE})))

`browser_snapshot` 은 지금 열려 있는 화면을 **접근성 트리 텍스트**로 받아 옵니다.
사진이 아니라 텍스트라 모델이 그대로 읽을 수 있고, 요소마다 붙은 `ref=e12` 같은 표식이
나중에 **클릭·입력할 때의 주소**가 됩니다.

In [ ]:
# 스냅샷 본문도 "### Result" 절이 아니다. 통째로 문자열로 뽑는다.
snapshot = block_text(await by_name["browser_snapshot"].ainvoke({}))
print("스냅샷 길이:", len(snapshot), "자")
print(snapshot[:800])

---
## 3. 스크롤하며 이미지 모으기

이 페이지는 **화면을 내려야 다음 이미지가 붙습니다**. 주소만으로는 12장이 전부입니다.
브라우저를 실제로 움직여야 나머지가 보인다는 것이 이 실습의 핵심입니다.

세 도구를 이어 씁니다.

- `browser_evaluate`: 우리가 준 자바스크립트를 **지금 열린 페이지 안에서** 실행해 값을 돌려준다
- `browser_press_key`: 사람이 키보드를 누르듯 키를 보낸다(`End` 는 화면을 맨 아래로)
- `browser_wait_for`: 정해진 시간만큼 기다린다(새 이미지를 받아 그릴 틈을 준다)

In [ ]:
# 페이지 안에서 실행할 자바스크립트. 브라우저가 들고 있는 지금 화면에 대고 직접 묻는다.
# 화살표 함수 하나를 문자열로 넘기면 서버가 그것을 페이지 안에서 실행하고 반환값을 돌려준다.
COUNT_IMAGES = "() => document.querySelectorAll('img').length"

# 스크롤하기 전의 이미지 수를 먼저 세어 기준을 잡는다.
print("처음 이미지 수:", result_value(await by_name["browser_evaluate"].ainvoke({"function": COUNT_IMAGES})))

In [ ]:
for step in range(1, 4):
    await by_name["browser_press_key"].ainvoke({"key": "End"})     # 화면을 맨 아래로
    await by_name["browser_wait_for"].ainvoke({"time": 2})         # 새 이미지가 그려질 틈을 준다
    counted = result_value(await by_name["browser_evaluate"].ainvoke({"function": COUNT_IMAGES}))
    print(f"{step}번째 스크롤 뒤 이미지 수:", counted)

In [ ]:
# 이번엔 개수가 아니라 주소를 꺼낸다. 같은 도구에 다른 자바스크립트를 넘기면 되는 일이 달라진다.
IMAGE_SOURCES = "() => Array.from(document.querySelectorAll('img')).map(img => img.src)"

# 모인 이미지 주소를 꺼내 앞의 몇 개만 눈으로 확인한다.
sources = result_value(await by_name["browser_evaluate"].ainvoke({"function": IMAGE_SOURCES}))
print(sources[:600])

### 🖐️ 함께 따라하기: 이미지 말고 상품 이름 모으기

데모는 `img` 요소의 주소를 모았습니다. 같은 화면에서 **다른 것**을 꺼내 봅니다.
이 페이지의 상품 이름은 `product-name` 클래스가 붙은 요소에 들어 있습니다.

1. 상품 이름을 모으는 자바스크립트를 문자열로 만드세요.

```
() => Array.from(document.querySelectorAll('.product-name')).map(el => el.textContent.trim())
```

2. `browser_evaluate` 의 `function` 인자로 넘겨 호출하고, `result_value()` 로 값을 꺼내 출력하세요.
3. 이어서 상품 **개수**만 세는 자바스크립트(`() => document.querySelectorAll('.product-name').length`)도 실행해 출력하세요.
4. 그 개수가 앞에서 센 **이미지 수와 같은지** 비교해 보세요.

**확인 기준**: 상품 이름이 여러 개 나오고, 개수는 앞 절에서 센 이미지 수와 같거나 비슷합니다(상품마다 이미지가 하나씩이므로).
스크롤을 한 뒤라 처음 12개보다 많아야 합니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) .product-name 요소의 textContent 를 모으는 자바스크립트 문자열을 만든다
# 2) browser_evaluate 의 function 인자로 넘겨 호출하고 result_value 로 값을 꺼내 출력한다
# 3) .product-name 의 개수만 세는 자바스크립트도 실행해 출력한다
# 4) 이미지 수와 비교한다

---
## 4. 에이전트에 붙이기: 필요한 도구만 골라서

도구 20여 개를 모두 넘기면 모델이 그 설명을 **전부 읽어야 해서** 비용이 커지고 선택도 흔들립니다.
이 일에 필요한 것만 골라 넘깁니다. **도구를 고르는 것도 설계**입니다.

In [ ]:
needed = ["browser_navigate", "browser_press_key", "browser_wait_for", "browser_evaluate"]
picked = [by_name[name] for name in needed]
print("에이전트에 넘길 도구:", needed)

model = ChatOpenAI(model="gpt-4o-mini", temperature=0)
agent = create_agent(
    model,
    picked,
    system_prompt=(
        "너는 웹 페이지에서 자료를 모으는 담당이다. "
        "먼저 browser_navigate 로 페이지를 열고, browser_press_key 로 End 키를 눌러 화면을 내린 뒤 "
        "browser_wait_for 로 2초쯤 기다려 새 내용이 그려질 시간을 준다. "
        "화면에 실제로 있는 값만 쓰고, 주소를 지어내지 않는다. "
        "이미지 주소는 browser_evaluate 에 "
        "\"() => Array.from(document.querySelectorAll('img')).map(img => img.src)\" 를 넘겨 확인한다."
    ),
)
print("에이전트 준비 완료")

질문에서 **세는 시점을 못 박습니다**. "처음 이미지 수" 만 적으면 모델이 세어 보지 않고 지어냅니다.
"두 수 모두 `browser_evaluate` 로 실제로 센 값이어야 한다" 처럼 **확인 방법까지** 적어 주는 것이 요령입니다.

In [ ]:
question = (
    f"{SITE} 를 열고, 스크롤하기 전에 먼저 이미지 수를 세어 둬. "
    "그다음 화면을 끝까지 세 번 내리고, 내릴 때마다 2초씩 기다린 뒤 이미지 수를 다시 세어 줘. "
    "마지막에 처음 수와 마지막 수, 그리고 상품 이미지 주소 5개를 목록으로 보여 줘. "
    "두 수 모두 browser_evaluate 로 실제로 센 값이어야 한다."
)
print("질문:", question, "\n")

result = await agent.ainvoke({"messages": question})
print_trajectory(result)

---
## 5. 세션 닫기

브라우저 서버는 우리가 띄운 **자식 프로세스**입니다. 닫지 않으면 커널이 살아 있는 동안 계속 떠 있습니다.
`async with` 를 직접 풀어 썼으니 **끝내는 쪽도 직접** 불러 줍니다.

In [ ]:
await ctx.__aexit__(None, None, None)   # 세션을 닫는다(= async with 블록을 빠져나가는 것과 같다)
print("세션을 닫았습니다. 이제 위 도구들은 쓸 수 없습니다.")

---
## 이번 실습 정리

| 배운 것 | 요점 |
|---|---|
| 상태 있는 서버 | 열어 둔 페이지가 남아야 하므로 세션을 **직접 열고 닫는다**(`__aenter__`/`__aexit__`) |
| 서버 인자 | `--isolated`·`--output-dir` 로 띄우는 방식을 고른다. 격리 프로필은 기본으로 쓰고, `--headless` 를 더하면 창 없이 돈다 |
| 화면 읽기 | `browser_snapshot` 은 접근성 트리 **텍스트**. `ref` 표식이 클릭 대상의 주소가 된다 |
| 페이지 안에서 묻기 | `browser_evaluate` 로 자바스크립트를 실행해 지금 화면의 값을 직접 얻는다 |
| 도구 고르기 | 20여 개를 다 넘기지 않고 **일에 필요한 것만**. 비용과 정확도가 함께 좋아진다 |
| 지어내기 방지 | 질문에 **확인 방법**(무엇으로 세었는지)을 적어 둔다 |

다음 실습: `04_SQLite_MCP.ipynb` 에서 **데이터베이스**를 붙여 에이전트가 스스로 SQL 을 쓰게 합니다.